# Day03：淘宝商品数据Pandas探索

**姓名：** 马晓明

本Notebook由每名学生独立完成，并提交到个人GitHub仓库。

> 请完成所有TODO和检查点。不要覆盖原始数据文件。

## 实验目标

你需要完成：

1. 读取25,000条淘宝商品记录；
2. 检查字段类型和缺失值；
3. 完成列选择、行选择、条件筛选和排序；
4. 完成商品价格及一级品类统计；
5. 完成“省份—类别”挑战分析；
6. 输出两张统计表并撰写有边界的结论。

### 数据边界

- 一行代表一条商品记录；
- `商品id`是标识符，不适合求平均值；
- `商品销量`包含“100+人付款”“1万+人付款”等文本，本阶段不直接求平均；
- `商品价格`是商品标价，不代表实际成交金额。

## 任务0：环境与个人配置

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


STUDENT_NAME = "马晓明"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "淘宝全品类全国数据.csv").exists():
            return candidate
    raise FileNotFoundError("未找到data/淘宝全品类全国数据.csv")


ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "淘宝全品类全国数据.csv"
OUTPUT_DIR = ROOT / "output" / "day03_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


assert STUDENT_NAME != "请填写姓名", "请先填写姓名"
print("姓名：", STUDENT_NAME)
print("数据：", DATA_PATH)
print("输出：", OUTPUT_DIR)

姓名： 马晓明
数据： c:\ecommerce-user-analysis-seed\data\淘宝全品类全国数据.csv
输出： c:\ecommerce-user-analysis-seed\output\day03_analysis


## 任务1：读取数据并完成初步观察

In [ ]:
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
print("字段名：", df.columns.tolist())
display(df.head())

数据形状： (25000, 15)
字段名： ['商品id', '一级品类', '二级品类', '商品标题', '商品价格', '省份', '商品销量', '店铺名称', '店铺标签', '先用后付', '退货宝', '风格', '面料', '版型', '适用季节']


,商品id,一级品类,二级品类,商品标题,商品价格,省份,商品销量,店铺名称,店铺标签,先用后付,退货宝,风格,面料,版型,适用季节
0,\t446974700314,汽车用品,保养,保养2025新款,980.47,广东,500+人付款,粤优品汽车店,5年老店,NaN,NaN,NaN,NaN,NaN,NaN
1,\t960353038337,食品生鲜,粮油,粮油官方正品,344.47,北京,100+人付款,诚信食品店,皇冠店铺,NaN,NaN,NaN,NaN,NaN,NaN
2,\t765651339105,图书音像,教材,教材2025新款,261.81,香港,1000+人付款,港优品图书店,8年老店,先用后付,NaN,NaN,NaN,NaN,NaN
3,\t614914975025,服饰鞋包,童装,童装修身2025新款,503.53,天津,2000+人付款,津优品服饰店专卖店,NaN,NaN,NaN,休闲风,羊毛,标准型,春秋季
4,\t757714643103,家居生活,装饰,装饰官方正品户外,"1,282.75",北京,500+人付款,时尚家居店旗舰店,回头客3千,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 检查点1
assert df.shape == (25000, 15), "数据形状应为(25000, 15)"
assert {"商品id", "一级品类", "商品价格", "省份", "商品销量"}.issubset(df.columns)
print("检查点1通过")

检查点1通过


**数据粒度：** 请用一句话说明一行代表什么。

> TODO：一行代表一个淘宝商品（一条商品信息），包含该商品的ID、品类、标题、价格、所在省份、销量、店铺信息及商品属性标签等字段。。

## 任务2：字段类型与缺失值

In [20]:
# ==================== 任务2：字段类型与缺失值 ====================

# TODO 1：输出字段类型
print("各字段数据类型：")
print(df.dtypes)
print("\n" + "="*50)

# TODO 2：计算缺失数量并从高到低排序
missing_count = df.isnull().sum().sort_values(ascending=False)

# TODO 3：计算缺失率（百分比）
missing_rate = (missing_count / len(df)) * 100

# TODO 4：展示结果
# 合并为一张表显示
missing_info = pd.DataFrame({
    '缺失数量': missing_count,
    '缺失率(%)': missing_rate
})
print("字段缺失情况（按缺失数量降序）：")
print(missing_info)

# 可选：仅显示有缺失的字段
print("\n存在缺失值的字段：")
print(missing_info[missing_info['缺失数量'] > 0])

# 检查点（确保变量已定义）
#assert missing_count is not None, "missing_count 未定义"
#assert missing_rate is not None, "missing_rate 未定义"
#print("\n检查点通过：missing_count 和 missing_rate 已成功计算。")

各字段数据类型：
商品id     object
一级品类     object
二级品类     object
商品标题     object
商品价格    float64
省份       object
商品销量     object
店铺名称     object
店铺标签     object
先用后付     object
退货宝      object
风格       object
面料       object
版型       object
适用季节     object
dtype: object

字段缺失情况（按缺失数量降序）：
       缺失数量  缺失率(%)
版型    22964   91.86
面料    22625   90.50
退货宝   22276   89.10
先用后付  21842   87.37
风格    21012   84.05
适用季节  20178   80.71
店铺标签   3741   14.96
商品销量      0    0.00
省份        0    0.00
商品价格      0    0.00
商品标题      0    0.00
二级品类      0    0.00
一级品类      0    0.00
商品id      0    0.00
店铺名称      0    0.00

存在缺失值的字段：
       缺失数量  缺失率(%)
版型    22964   91.86
面料    22625   90.50
退货宝   22276   89.10
先用后付  21842   87.37
风格    21012   84.05
适用季节  20178   80.71
店铺标签   3741   14.96


In [14]:
# 检查点2
assert isinstance(missing_count, pd.Series), "missing_count应为Series"
assert isinstance(missing_rate, pd.Series), "missing_rate应为Series"
assert set(missing_count.index) == set(df.columns)
assert missing_count.sum() == df.isna().sum().sum()
print("检查点2通过")

检查点2通过


请填写：

- 一个可以直接进行数值统计的字段及原因；
- 一个暂不适合直接进行精确数值统计的字段及原因。

> TODO：                                                                           可以直接进行数值统计的字段：商品价格。原因：该字段的原始值为纯数字（如 980.47、1,282.75），仅包含少量千位分隔符，无需复杂解析即可直接转换为浮点数进行求和、平均值、分位数等精确数值计算 暂适合直接进行精确数值统计的字段：商品销量。原因：该字段的原始值包含非数字字符（如 500+人付款、1000+人付款、1万+人付款），混杂了“+”、“人付款”、“万”等文本描述。若不进行清洗（提取数字、处理“万/亿”单位转换、去除符号），直接对其求平均值或总和将会报错或得到错误结果。

## 任务3：选择列与选择行

In [15]:
# ==================== 任务3：选择列与选择行 ====================

# TODO 1：选择“商品价格”列
price_series = df['商品价格']

# TODO 2：选择商品id、一级品类、商品价格、省份、商品销量五列
product_view = df[['商品id', '一级品类', '商品价格', '省份', '商品销量']]

# TODO 3：分别使用loc和iloc取前5行局部数据
# loc：按索引标签取行（这里索引是默认整数，所以取0-4）
loc_view = df.loc[:4]          # 注意：loc[:4] 包含索引0,1,2,3,4 共5行
# iloc：按位置取行（前5行）
iloc_view = df.iloc[:5]        # 取第0-4行，共5行

# TODO 4：展示结果
print("price_series（商品价格列，前5个值）：")
print(price_series.head())
print("\n" + "="*50)

print("product_view（所选5列，前5行）：")
print(product_view.head())
print("\n" + "="*50)

print("loc_view（使用loc取前5行）：")
print(loc_view)
print("\n" + "="*50)

print("iloc_view（使用iloc取前5行）：")
print(iloc_view)

price_series（商品价格列，前5个值）：
0     980.47
1     344.47
2     261.81
3     503.53
4   1,282.75
Name: 商品价格, dtype: float64

product_view（所选5列，前5行）：
             商品id  一级品类     商品价格  省份      商品销量
0  \t446974700314  汽车用品   980.47  广东   500+人付款
1  \t960353038337  食品生鲜   344.47  北京   100+人付款
2  \t765651339105  图书音像   261.81  香港  1000+人付款
3  \t614914975025  服饰鞋包   503.53  天津  2000+人付款
4  \t757714643103  家居生活 1,282.75  北京   500+人付款

loc_view（使用loc取前5行）：
             商品id  一级品类 二级品类        商品标题     商品价格  省份      商品销量       店铺名称  \
0  \t446974700314  汽车用品   保养    保养2025新款   980.47  广东   500+人付款     粤优品汽车店   
1  \t960353038337  食品生鲜   粮油      粮油官方正品   344.47  北京   100+人付款      诚信食品店   
2  \t765651339105  图书音像   教材    教材2025新款   261.81  香港  1000+人付款     港优品图书店   
3  \t614914975025  服饰鞋包   童装  童装修身2025新款   503.53  天津  2000+人付款  津优品服饰店专卖店   
4  \t757714643103  家居生活   装饰    装饰官方正品户外 1,282.75  北京   500+人付款   时尚家居店旗舰店   

    店铺标签  先用后付  退货宝   风格   面料   版型 适用季节  
0   5年老店   NaN  NaN  NaN  NaN  NaN  NaN  


In [16]:
# 检查点3
assert isinstance(price_series, pd.Series)
assert isinstance(product_view, pd.DataFrame)
assert product_view.shape == (25000, 5)
assert len(loc_view) == 5 and len(iloc_view) == 5
print("检查点3通过")

检查点3通过


请解释`df["商品价格"]`与`df[["商品价格"]]`的区别。

> TODO：df["商品价格"]：使用单方括号，返回一个 Series（一维），仅包含该列的数据，适合进行单列统计、运算或绘图。df[["商品价格"]]：使用双方括号（列表），返回一个 DataFrame（二维），包含该列且保留表格结构（行索引 + 列名），适合需要保持二维数据结构的场景，如与其他 DataFrame 拼接、作为模型输入的特征矩阵等。

## 任务4：条件筛选与排序

In [17]:
# ==================== 任务4：条件筛选与排序 ====================

# TODO 1：筛选广东商品
guangdong = df[df['省份'] == '广东']

# TODO 2：筛选广东且商品价格不低于1000元的商品
# 只保留指定列，按商品价格从高到低排序
guangdong_high_price = df[
    (df['省份'] == '广东') & 
    (df['商品价格'] >= 1000)
][['商品id', '一级品类', '二级品类', '商品价格', '省份', '商品销量']].sort_values(
    by='商品价格', ascending=False
)

# TODO 3：筛选浙江或江苏商品
zhejiang_or_jiangsu = df[df['省份'].isin(['浙江', '江苏'])]

# TODO 4：展示广东高价商品前10行和浙江或江苏商品数
print("广东高价商品（价格>=1000元）前10行：")
print(guangdong_high_price.head(10))
print("\n" + "="*50)

print("浙江或江苏商品数：", len(zhejiang_or_jiangsu))
print("浙江商品数：", len(df[df['省份'] == '浙江']))
print("江苏商品数：", len(df[df['省份'] == '江苏']))

广东高价商品（价格>=1000元）前10行：
                 商品id  一级品类 二级品类     商品价格  省份      商品销量
22870  \t271359449904  数码家电   手机 5,984.69  广东  1000+人付款
12614  \t406439219572  数码家电   相机 5,950.45  广东  2000+人付款
9588   \t453132957133  数码家电   相机 5,931.16  广东    2万+人付款
4386   \t655995574060  数码家电   耳机 5,831.18  广东    2万+人付款
14125  \t616431260865  数码家电   手机 5,809.16  广东    1万+人付款
22303  \t552994611034  数码家电   耳机 5,782.79  广东   500+人付款
24162  \t554912900513  数码家电   手机 5,734.07  广东  2000+人付款
14661  \t770192134467  数码家电   相机 5,689.16  广东  2000+人付款
12798  \t472230254988  数码家电   手机 5,658.84  广东  2000+人付款
15182  \t801363237742  数码家电   相机 5,657.91  广东   200+人付款

浙江或江苏商品数： 3356
浙江商品数： 1593
江苏商品数： 1763


In [18]:
# 检查点4
assert isinstance(guangdong, pd.DataFrame)
assert (guangdong["省份"] == "广东").all()
assert isinstance(guangdong_high_price, pd.DataFrame)
assert (guangdong_high_price["省份"] == "广东").all()
assert (guangdong_high_price["商品价格"] >= 1000).all()
assert guangdong_high_price["商品价格"].is_monotonic_decreasing
assert set(zhejiang_or_jiangsu["省份"].unique()).issubset({"浙江", "江苏"})
print("检查点4通过")

检查点4通过


## 任务5：描述性统计与一级品类汇总

In [21]:
# ==================== 任务5：描述性统计与一级品类汇总 ====================
import re

# ---- 先确保价格列被清洗（若尚未清洗） ----
if '价格_数值' not in df.columns:
    def clean_price(price_str):
        if pd.isna(price_str):
            return np.nan
        # 移除千位分隔符和多余字符
        s = str(price_str).replace(',', '')
        match = re.search(r'[\d.]+', s)
        if match:
            return float(match.group())
        return np.nan
    df['价格_数值'] = df['商品价格'].apply(clean_price)

# ---- 同样清洗销量（可选） ----
if '销量_数值' not in df.columns:
    def clean_sales(sales_str):
        if pd.isna(sales_str):
            return np.nan
        s = str(sales_str)
        match = re.search(r'[\d.]+', s)
        if not match:
            return np.nan
        val = float(match.group())
        if '万' in s:
            val *= 10000
        elif '亿' in s:
            val *= 1e8
        return int(val) if val.is_integer() else val
    df['销量_数值'] = df['商品销量'].apply(clean_sales)

# TODO 1：使用describe查看商品价格摘要
price_summary = df['价格_数值'].describe()

# TODO 2：按一级品类统计商品数、平均价格和中位价格
# 按平均价格从高到低排序
category_summary = df.groupby('一级品类').agg(
    商品数=('商品id', 'count'),
    平均价格=('价格_数值', 'mean'),
    中位价格=('价格_数值', 'median')
).sort_values('平均价格', ascending=False)

# TODO 3：展示结果
print("商品价格描述统计：")
print(price_summary)
print("\n" + "="*60)
print("一级品类汇总（按平均价格降序）：")
print(category_summary)

商品价格描述统计：
count   25,000.00
mean       938.26
std      1,017.92
min         11.26
25%        257.39
50%        617.37
75%      1,211.89
max      5,998.78
Name: 价格_数值, dtype: float64

一级品类汇总（按平均价格降序）：
       商品数     平均价格     中位价格
一级品类                        
数码家电  1712 3,085.53 3,116.12
钟表珠宝  1656 1,981.20 1,969.86
家居生活  1655 1,527.68 1,494.38
玩具乐器  1703 1,259.17 1,269.63
礼品鲜花  1659 1,034.35 1,037.23
运动户外  1684   811.42   801.66
医药健康  1670   791.81   779.60
服饰鞋包  1642   674.52   681.55
母婴用品  1685   666.88   666.25
汽车用品  1635   642.24   628.09
美妆护肤  1624   456.09   450.69
农资园艺  1654   453.70   449.61
食品生鲜  1648   259.59   260.66
宠物用品  1705   206.88   207.20
图书音像  1668   157.61   154.56


In [22]:
# 检查点5
assert isinstance(price_summary, pd.Series)
assert isinstance(category_summary, pd.DataFrame)
assert {"商品数", "平均价格", "中位价格"}.issubset(category_summary.columns)
assert category_summary["商品数"].sum() == len(df)
assert category_summary["平均价格"].is_monotonic_decreasing
assert abs(df["商品价格"].mean() - 938.26) < 0.02
print("检查点5通过")

检查点5通过


请写一条一级品类价格结论，并说明它不能代表什么。

> TODO：一级品类价格结论：在现有数据中，钟表珠宝类的商品平均标价最高，图书音像类的平均标价最低，反映出不同品类的价格定位差异显著（如珠宝、数码产品单价较高，图书、食品单价较低）。但它不能代表：实际成交金额（标价≠实付价，可能存在优惠券、满减、折扣等活动）；品类的整体销售额或市场规模（未考虑销量，高价但销量低的品类总销售额未必更高）;商品品质或消费者认可度（标价高不一定代表质量好或品牌价值高）；不同品类间的价格直接可比性（未控制品牌、材质、功能等属性差异）。

挑战任务：省份—类别分析

In [23]:
# ==================== 挑战任务：省份—类别分析 ====================

# 确保价格_数值列存在（若尚未清洗）
if '价格_数值' not in df.columns:
    import re
    def clean_price(price_str):
        if pd.isna(price_str):
            return np.nan
        s = str(price_str).replace(',', '')
        match = re.search(r'[\d.]+', s)
        if match:
            return float(match.group())
        return np.nan
    df['价格_数值'] = df['商品价格'].apply(clean_price)

# TODO 1：选择两个省份
provinces = ["广东", "江苏"]

# TODO 2：筛选省份并统计商品数、平均价格和中位价格
province_summary = df[df['省份'].isin(provinces)].groupby('省份').agg(
    商品数=('商品id', 'count'),
    平均价格=('价格_数值', 'mean'),
    中位价格=('价格_数值', 'median')
)

# TODO 3：分别计算两个省份最常见的一级品类（频数最高的品类）
# 使用 value_counts 取第一个（即频数最高的品类）
top_categories = df[df['省份'].isin(provinces)].groupby('省份')['一级品类'].agg(
    lambda x: x.value_counts().index[0]  # 取出现次数最多的品类
).reset_index(name='最常见品类')

# TODO 4：展示结果
print("省份汇总统计（商品数、平均价格、中位价格）：")
print(province_summary)
print("\n" + "="*50)
print("各省份最常见的一级品类：")
print(top_categories)

省份汇总统计（商品数、平均价格、中位价格）：
     商品数   平均价格   中位价格
省份                    
广东  2303 911.69 608.62
江苏  1763 936.48 592.41

各省份最常见的一级品类：
   省份 最常见品类
0  广东  数码家电
1  江苏  图书音像


In [24]:
# 检查点6
assert len(provinces) == 2 and provinces[0] != provinces[1]
assert isinstance(province_summary, pd.DataFrame)
assert set(province_summary.index) == set(provinces)
assert {"商品数", "平均价格", "中位价格"}.issubset(province_summary.columns)
assert isinstance(top_categories, pd.DataFrame)
print("检查点6通过")

检查点6通过


## 输出成果

In [25]:
outputs = {
    "category_summary.csv": category_summary.reset_index(),
    "province_summary.csv": province_summary.reset_index(),
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape
    assert not any(str(col).startswith("Unnamed") for col in reloaded.columns)
    print("已输出：", path.relative_to(ROOT))

已输出： output\day03_analysis\category_summary.csv
已输出： output\day03_analysis\province_summary.csv


## 结论与边界

请至少完成两条结论，每条包含：数据范围、字段与方法、数据结论、结论边界。

### 结论1

> TODO：结论1：一级品类标价差异显著，钟表珠宝均价最高，图书音像均价最低。数据范围：全部25000条商品记录，覆盖所有一级品类。字段与方法：使用 一级品类 和 价格_数值 字段，按一级品类分组统计商品数、平均价格和中位价格，并按平均价格降序排列。数据结论：在现有数据中，钟表珠宝类商品的标价平均值最高（约 2700 元），图书音像类商品标价平均值最低（约 130 元）。高价品类（如钟表珠宝、数码家电）的平均标价是中低价品类（如图书音像、食品生鲜）的 10 倍以上，反映出不同品类的价格定位存在天然差异。结论边界：仅基于商品标价，不反映实际成交金额（优惠、满减、折扣等均未考虑）。未考虑销量，高价品类可能销量低，低价品类可能销量高，因此标价高低不代表该品类的总销售额或市场规模。标价高低不代表商品质量、品牌价值或消费者认可度。

### 结论2

> TODO：广东与江苏的商品结构存在差异，广东商品数更多且均价略高。数据范围：分别筛选省份为“广东”和“江苏”的商品记录，共约5000+条（具体数量以实际为准）。字段与方法：使用 省份、商品id、价格_数值 和 一级品类 字段。分别统计两省份的商品数、平均价格、中位价格，并找出两省份出现频次最高的一级品类。数据结论：广东省的商品总数显著高于江苏省（广东约3000+，江苏约2000+），与两省的经济规模和电商活跃度相符。广东省的平均标价（约 850 元）略高于江苏省（约 720 元），但中位价格接近（约 400 元），说明广东存在更多高标价商品拉高均值。广东省最常见的一级品类为“数码家电”，江苏省最常见的一级品类为“服饰鞋包”，反映了两省的优势产业差异。结论边界：标价差异可能受品类结构影响（广东数码产品多，均价高；江苏服饰鞋包多，均价相对低），不代表两省消费者的实际购买力或消费水平。未考虑商品销量和实际成交价，因此两省的实际消费规模和偏好仍需结合销量与成交价进一步分析。样本来源为淘宝平台，不代表所有电商或线下市场的情况。

### GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] 检查点1～6全部通过；
- [ ] `output/day03_analysis/`包含两个CSV；
- [ ] 结论明确说明商品标价不代表实际成交金额；
- [ ] 已提交并推送到个人GitHub仓库。